In [1]:
import requests
import pandas as pd
import time
from datetime import datetime


# -------------------------------
# Helper: Retry logic
# -------------------------------
def make_request(url, params, city_name):
    for attempt in range(3):
        try:
            response = requests.get(url, params=params, timeout=10)

            if response.status_code != 200:
                raise Exception(f"HTTP {response.status_code}")

            data = response.json()

            return data

        except Exception as e:
            print(f"[{city_name}] Attempt {attempt+1} failed: {e}")

            # Exponential backoff: 1s → 2s → 4s
            time.sleep(2 ** attempt)

    raise Exception(f"[{city_name}] Failed after 3 retries")


# -------------------------------
# Fetch Historical Data
# -------------------------------
def fetch_historical(city_name, latitude, longitude, start_date, end_date, variables):
    
    # ✅ Date validation
    if start_date >= end_date:
        raise ValueError("start_date must be before end_date")

    today = datetime.today().strftime("%Y-%m-%d")
    if end_date > today:
        raise ValueError("end_date cannot be in the future")

    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "daily": ",".join(variables)
    }

    data = make_request(url, params, city_name)

    # ✅ Check response
    if "daily" not in data:
        raise ValueError(f"[{city_name}] Malformed response: {data}")

    df = pd.DataFrame(data["daily"])

    # ✅ Basic validation
    if df.empty:
        raise ValueError(f"[{city_name}] Empty dataset returned")

    df["time"] = pd.to_datetime(df["time"])
    df["city"] = city_name

    return df


# -------------------------------
# Fetch Forecast Data
# -------------------------------
def fetch_forecast(city_name, latitude, longitude, variables):

    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "daily": ",".join(variables)
    }

    data = make_request(url, params, city_name)

    if "daily" not in data:
        raise ValueError(f"[{city_name}] Malformed forecast response: {data}")

    df = pd.DataFrame(data["daily"])

    if df.empty:
        raise ValueError(f"[{city_name}] Empty forecast data")

    df["time"] = pd.to_datetime(df["time"])
    df["city"] = city_name

    return df


# -------------------------------
# Fetch All Cities
# -------------------------------
def fetch_all_cities(cities_config, start_date, end_date, variables):

    all_data = {}

    for city in cities_config:
        name = city["name"]
        lat = city["latitude"]
        lon = city["longitude"]

        print(f"Fetching historical data for {name}...")

        df = fetch_historical(
            name,
            lat,
            lon,
            start_date,
            end_date,
            variables
        )

        all_data[name] = df

        print(f"{name}: {len(df)} rows")

    return all_data

In [2]:
import sys
import os

sys.path.append(os.path.abspath("../src"))

from ingestion import fetch_historical

df = fetch_historical(
    "Baku",
    40.41,
    49.87,
    "2020-01-01",
    "2022-12-31",
    ["temperature_2m_max", "precipitation_sum"]
)

df.head()

,time,temperature_2m_max,precipitation_sum,city
0,2020-01-01,11.4,0.0,Baku
1,2020-01-02,8.6,0.3,Baku
2,2020-01-03,7.6,1.4,Baku
3,2020-01-04,8.4,0.2,Baku
4,2020-01-05,7.7,2.6,Baku


In [3]:
import sys
import os

sys.path.append(os.path.abspath("../src"))

from ingestion import fetch_historical
from config import CITIES, START_DATE, END_DATE, DAILY_VARIABLES

city = CITIES[0]

df = fetch_historical(
    city_name=city["name"],
    latitude=city["latitude"],
    longitude=city["longitude"],
    start_date=START_DATE,
    end_date=END_DATE,
    variables=DAILY_VARIABLES
)

df.head()

,time,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max,apparent_temperature_max,relative_humidity_2m_mean,weather_code,city
0,2020-01-01,11.4,4.4,0.0,11.4,8.8,88,3,Baku
1,2020-01-02,8.6,2.7,0.3,33.0,3.2,86,51,Baku
2,2020-01-03,7.6,3.4,1.4,21.7,3.3,84,51,Baku
3,2020-01-04,8.4,5.7,0.2,22.7,4.6,81,51,Baku
4,2020-01-05,7.7,5.8,2.6,12.6,6.4,94,53,Baku


In [4]:
print(df.shape)
print(df.columns)
print(df.dtypes)
print(df["time"].min(), df["time"].max())

(2192, 9)
Index(['time', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum',
       'wind_speed_10m_max', 'apparent_temperature_max',
       'relative_humidity_2m_mean', 'weather_code', 'city'],
      dtype='object')
time                         datetime64[ns]
temperature_2m_max                  float64
temperature_2m_min                  float64
precipitation_sum                   float64
wind_speed_10m_max                  float64
apparent_temperature_max            float64
relative_humidity_2m_mean             int64
weather_code                          int64
city                                 object
dtype: object
2020-01-01 00:00:00 2025-12-31 00:00:00


In [5]:
from ingestion import fetch_all_cities

data = fetch_all_cities(
    cities_config=CITIES,
    start_date=START_DATE,
    end_date=END_DATE,
    variables=DAILY_VARIABLES
)

for city_name, df in data.items():
    print(f"{city_name}: {df.shape}, {df['time'].min()} → {df['time'].max()}")

Baku: (2192, 9), 2020-01-01 00:00:00 → 2025-12-31 00:00:00
Lankaran: (2192, 9), 2020-01-01 00:00:00 → 2025-12-31 00:00:00
Guba: (2192, 9), 2020-01-01 00:00:00 → 2025-12-31 00:00:00
Gabala: (2192, 9), 2020-01-01 00:00:00 → 2025-12-31 00:00:00
Shaki: (2192, 9), 2020-01-01 00:00:00 → 2025-12-31 00:00:00


In [10]:
import os
from ingestion import fetch_all_cities
from config import CITIES, START_DATE, END_DATE, DAILY_VARIABLES

# 1. Fetch data
data = fetch_all_cities(
    cities_config=CITIES,
    start_date=START_DATE,
    end_date=END_DATE,
    variables=DAILY_VARIABLES
)

# 2. Quick check
for city_name, df in data.items():
    print(f"""
City: {city_name}
Rows: {len(df)}
Date range: {df['time'].min()} → {df['time'].max()}
Missing values:
{df.isna().sum()}
""")

# 3. Create output folder
os.makedirs("../data/raw/historical", exist_ok=True)

# 4. Save each city as Parquet
for city_name, df in data.items():
    file_path = f"../data/raw/historical/{city_name.lower()}_2020_2025.parquet"
    df.to_parquet(file_path, index=False)
    print(f"Saved: {file_path}")


City: Baku
Rows: 2192
Date range: 2020-01-01 00:00:00 → 2025-12-31 00:00:00
Missing values:
time                         0
temperature_2m_max           0
temperature_2m_min           0
precipitation_sum            0
wind_speed_10m_max           0
apparent_temperature_max     0
relative_humidity_2m_mean    0
weather_code                 0
city                         0
dtype: int64


City: Lankaran
Rows: 2192
Date range: 2020-01-01 00:00:00 → 2025-12-31 00:00:00
Missing values:
time                         0
temperature_2m_max           0
temperature_2m_min           0
precipitation_sum            0
wind_speed_10m_max           0
apparent_temperature_max     0
relative_humidity_2m_mean    0
weather_code                 0
city                         0
dtype: int64


City: Guba
Rows: 2192
Date range: 2020-01-01 00:00:00 → 2025-12-31 00:00:00
Missing values:
time                         0
temperature_2m_max           0
temperature_2m_min           0
precipitation_sum            0
wind_sp

In [11]:
from ingestion import fetch_forecast

import os
os.makedirs("../data/raw/forecast", exist_ok=True)

for city in CITIES:
    df_f = fetch_forecast(
        city["name"],
        city["latitude"],
        city["longitude"],
        DAILY_VARIABLES
    )

    file_path = f"../data/raw/forecast/{city['name'].lower()}_forecast.parquet"
    df_f.to_parquet(file_path, index=False)

    print(f"Saved forecast: {file_path}")

Saved forecast: ../data/raw/forecast/baku_forecast.parquet
Saved forecast: ../data/raw/forecast/lankaran_forecast.parquet
Saved forecast: ../data/raw/forecast/guba_forecast.parquet
Saved forecast: ../data/raw/forecast/gabala_forecast.parquet
Saved forecast: ../data/raw/forecast/shaki_forecast.parquet
